In [0]:
from pyspark.sql import functions as F

from delta.tables import DeltaTable

# ---------------- CONFIG ----------------

BRONZE_JIRA  = "workspace.slainte_bronze.jira_issues_bronze"

GOLD_DB      = "workspace.slainte_gold"

DIM_EMPLOYEE = f"{GOLD_DB}.dim_employee"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD DIM ----------------

df_dim_employee = (

    spark.table(BRONZE_JIRA)

    .select(

        F.col("user_id").alias("source_user_id"),

        F.col("project_key").alias("project"),

        F.col("assignee").alias("employee_name")

    )

    .filter(

        F.col("user_id").isNotNull() &

        F.col("project_key").isNotNull() &

        F.col("assignee").isNotNull()

    )

    .distinct()

    .withColumn("employee_id", F.monotonically_increasing_id() + 1)

    .select("employee_id", "source_user_id", "project", "employee_name")

)

# ---------------- WRITE (MERGE) ----------------

if not spark.catalog.tableExists(DIM_EMPLOYEE):

    df_dim_employee.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(DIM_EMPLOYEE)

else:

    target = DeltaTable.forName(spark, DIM_EMPLOYEE)

    (

        target.alias("t")

        .merge(

            df_dim_employee.alias("s"),

            "t.source_user_id = s.source_user_id AND "

            "t.project = s.project AND "

            "t.employee_name = s.employee_name"

        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()

    )

print("✅ dim_employee created/merged successfully")
 